# Контрольные вопросы по лекции №9. Объединение, связывание и изменение формы данных

In [1]:
# Подготовка зависимостей
import numpy as np
import pandas as pd

1. Что такое конкатенация данных?

**Ответ**: Конкатенация (concatenation) в библиотеке pandas представляет собой процесс объединения данных, расположенных в двух или более объектах, в новый объект.

Конкатенация объектов Series просто приводит к созданию нового объекта Series с последовательно скопированными значениями.

Процесс конкатенации объектов DataFrame более сложен. Конкатенацию можно применять к любой оси указанных объектов. Вдоль заданной оси осуществляется логическая операция соединения индексных меток. Затем вдоль противоположной оси происходит выравнивание меток и заполнение пропущенных значений.

1.1 Понимание семантики конкатенации, принятой по умолчанию. Привести **свои** примеры.

**Ответ**: Конкатенацию можно выполнить с помощью функции библиотеки pandas `pd.concat()`. Общий синтаксис, выполняющий конкатенацию данных, состоит в том, чтобы передать список объектов, которые будут конкатенированы.

Вышеприведенный программный код присоединяет метки индекса и значения объекта s2 к конечной метке индекса и конечному значению объекта s1. В результате получаем дублирующиеся индексные метки, поскольку в ходе этой операции выравнивание не выполняется.

По умолчанию строки добавляются по порядку и в результате мы можем получить дублирующиеся индексные метки вдоль индекса строк. Итоговый набор меток столбцов получается в результате объединения индексных меток в указанных нами объектах DataFrame.

Если в обрабатываемом объекте DataFrame отсутствует столбец, но при этом он присутствует в другом обрабатываемом объекте DataFrame, то в итоговом датафрейме отсутствующие значения столбца будут заполнены пропусками.

Каждой группе данных в итоговом датафрейме можно дать свое название, воспользовавшись параметром `keys`. Он создает иерархический индекс объекта DataFrame, который позволяет отдельно обращаться к каждой группе данных с помощью свойства `.loc` объекта DataFrame.

In [2]:
# создаем два объекта DataFrame для конкатенации,
# используя те же самые индексные метки и имена столбцов,
# но другие значения
df1 = pd.DataFrame(np.arange(9).reshape(3, 3),
                   columns=['a', 'b', 'c'])
# df2 имеет значения 9 .. 18
df2 = pd.DataFrame(np.arange(9, 18).reshape(3, 3),
                   columns=['a', 'b', 'c'])
print(df1)
print(df2)
# выполняем конкатенацию
print(pd.concat([df1, df2]))

# демонстрируем конкатенацию двух объектов DataFrame
# с разными столбцами
df1 = pd.DataFrame(np.arange(9).reshape(3, 3),
                   columns=['a', 'b', 'c'])
df2 = pd.DataFrame(np.arange(9, 18).reshape(3, 3),
                   columns=['a', 'c', 'd'])
# выполняем конкатенацию, пропусками будут заполнены
# значения столбца d в датафрейме df1 и
# значения столбца b в датафрейме df2
print(pd.concat([df1, df2], sort=True))

# выполняем конкатенацию двух объектов,
# но при этом создаем индекс с помощью
# заданных ключей
c = pd.concat([df1, df2], keys=['df1', 'df2'], sort=True)
# обратите внимание на метки строк в выводе
print(c)

   a  b  c
0  0  1  2
1  3  4  5
2  6  7  8
    a   b   c
0   9  10  11
1  12  13  14
2  15  16  17
    a   b   c
0   0   1   2
1   3   4   5
2   6   7   8
0   9  10  11
1  12  13  14
2  15  16  17
    a    b   c     d
0   0  1.0   2   NaN
1   3  4.0   5   NaN
2   6  7.0   8   NaN
0   9  NaN  10  11.0
1  12  NaN  13  14.0
2  15  NaN  16  17.0
        a    b   c     d
df1 0   0  1.0   2   NaN
    1   3  4.0   5   NaN
    2   6  7.0   8   NaN
df2 0   9  NaN  10  11.0
    1  12  NaN  13  14.0
    2  15  NaN  16  17.0


1.2 Переключение осей выравнивания. Привести **свои** примеры.

**Ответ**: Функция `pd.concat()` позволяет задать ось, к которой нужно применить выравнивание во время конкатенации. Следующий программный код конкатенирует два объекта DataFrame по оси столбцов, а выравнивание осуществляется по индексу строк. При этом явно задавать то или иное значение параметра sort не нужно, потому что ось строк, не участвующая в конкатенации, выровнена (датафреймы имеют одинаковое количество строк).

В результате получаем дублирующиеся столбцы. Это связано с тем, что в ходе конкатенации сначала происходит выравнивание по меткам индекса строк каждого объекта DataFrame, а затем осуществляется заполнение значений столбцов для первого объекта DataFrame и второго объекта DataFrame независимо от меток индекса строк.

Поскольку выравнивание осуществлялось по меткам индекса строк, мы получаем дублирующиеся столбцы. Значения для меток строк 0 и 1 в столбцах a и d, соответствующих датафрейму df3, заполняются пропусками, поскольку эти метки есть только в датафрейме df1. Значения для меток строк 3 и 4 в столбцах a, b и c, соответствующих датафрейму df1, заполняются пропусками, поскольку эти метки есть только в датафрейме df3.

In [3]:
df1 = pd.DataFrame(np.arange(9).reshape(3, 3),
                   columns=['a', 'b', 'c'])
# создаем новый датафрейм df3, чтобы конкатенировать его
# с датафреймом df1
# датафрейм df3 имеет общую с датафреймом df1
# метку 2 и общий столбец a
df3 = pd.DataFrame(np.arange(20, 26).reshape(3, 2),
                   columns=['a', 'd'],
                   index=[2, 3, 4])
print(df3)
# конкатенируем их по оси столбцов. Происходит выравнивание по меткам строк,
# осуществляется заполнение значений столбцов df1, а затем
# столбцов df3, получаем дублирующиеся столбцы
print(pd.concat([df1, df3], axis=1))

    a   d
2  20  21
3  22  23
4  24  25
     a    b    c     a     d
0  0.0  1.0  2.0   NaN   NaN
1  3.0  4.0  5.0   NaN   NaN
2  6.0  7.0  8.0  20.0  21.0
3  NaN  NaN  NaN  22.0  23.0
4  NaN  NaN  NaN  24.0  25.0


1.3 Определение типа соединения. Привести **свои** примеры на **каждый тип** соединения.

**Ответ**: По умолчанию конкатенация выполняет операцию внешнего соединения (outer join) индексных меток по оси, противоположной оси конкатенации (индексу строк). Итоговый набор меток представляет собой результат объединения этих меток.

Тип соединения можно изменить на внутреннее соединение (inner join), указав `join='inner'` в качестве параметра. Внутреннее соединение вместо объединения выполняет логическую операцию пересечения индексных меток.

Кроме того, можно разметить группы данных вдоль столбцов, применив параметр `keys` при выполнении конкатенации по оси столбцов (`axis=1`). Отобрать различные группы данных можно с помощью свойства `.loc` и создания срезов.

In [4]:
df1 = pd.DataFrame(np.arange(9).reshape(3, 3),
                   columns=['a', 'b', 'c'])
df3 = pd.DataFrame(np.arange(20, 26).reshape(3, 2),
                   columns=['a', 'd'],
                   index=[2, 3, 4])
# выполняем внутреннее соединение вместо внешнего
# результат представлен в виде одной строки
print(pd.concat([df1, df3], axis=1, join='inner'))

df2 = pd.DataFrame(np.arange(9, 18).reshape(3, 3),
                   columns=['a', 'c', 'd'])
# добавляем ключи к столбцам
df = pd.concat([df1, df2], axis=1, keys=['df1', 'df2'])
print(df)
# извлекаем данные из датафрейма
# с помощью ключа 'df2'
print(df.loc[:, 'df2'])

   a  b  c   a   d
2  6  7  8  20  21
  df1       df2        
    a  b  c   a   c   d
0   0  1  2   9  10  11
1   3  4  5  12  13  14
2   6  7  8  15  16  17
    a   c   d
0   9  10  11
1  12  13  14
2  15  16  17


1.4 Присоединение вместо конкатенации. Привести **свои** примеры.

**Ответ**: Кроме того, можно воспользоваться методом `.append()` объекта DataFrame (и Series), который объединяет два указанных объекта DataFrame по меткам индекса строк. Помним, что ось столбцов, не участвующая в конкатенации, не выровнена (датафреймы имеют неидентичный набор столбцов), явно задаем то или иное значение параметра sort, чтобы избежать предупреждения.

Как и при конкатенации по оси строк (`axis=0`), при выполнении присоединения индексные метки строк копируются без учета того, что произойдет их дублирование, а метки столбцов объединяются таким способом, который не гарантирует отсутствия в итоговом датафрейме столбцов с дублирующимися именами.

In [5]:
df1 = pd.DataFrame(np.arange(9).reshape(3, 3),
                   columns=['a', 'b', 'c'])
df2 = pd.DataFrame(np.arange(9, 18).reshape(3, 3),
                   columns=['a', 'c', 'd'])
# присоединяем df2 к df1
# Ошибка: в Pandas 2.0 метод .append() был окончательно удален (ранее он был помечен как устаревший)
# Вместо df1.append(df2) -> pd.concat([df1, df2])
pd.concat([df1, df2], sort=True)

,a,b,c,d
0,0,1.0,2,NaN
1,3,4.0,5,NaN
2,6,7.0,8,NaN
0,9,NaN,10,11.0
1,12,NaN,13,14.0
2,15,NaN,16,17.0


1.5 Игнорирование меток индекса. Привести **свои** примеры.

**Ответ**: Если вы хотите избавиться от дублирования меток в итоговом индексе и при этом сохранить все строки, вы можете воспользоваться параметром `ignore_index=True`. Это по сути возвращает тот же результат, за исключением того, что теперь наш индекс имеет тип Int64Index.

In [6]:
df1 = pd.DataFrame(np.arange(9).reshape(3, 3),
                   columns=['a', 'b', 'c'])
df2 = pd.DataFrame(np.arange(9, 18).reshape(3, 3),
                   columns=['a', 'b', 'c'])
# конкатенация с игнорированием меток индекса
print(pd.concat([df1, df2], ignore_index=True))

    a   b   c
0   0   1   2
1   3   4   5
2   6   7   8
3   9  10  11
4  12  13  14
5  15  16  17


2. Слияние данных, расположенных в нескольких объектах. Привести **свои** примеры.

**Ответ**: Библиотека pandas позволяет выполнить слияние объектов с помощью операций, аналогичных операциям соединения для баз данных, используя функцию `pd.merge()` и метод `.merge()` объекта DataFrame.

Процедура слияния объединяет данные двух объектов путем поиска совпадающих значений в одном или нескольких индексах столбцов или строк. Затем, применив семантику соединения к этим значениям, она возвращает новый объект — комбинацию данных обоих объектов.

Слияние полезно, поскольку оно позволяет нам создавать отдельный объект DataFrame для каждого типа данных, а также связывать данные в разных объектах DataFrame с помощью значений, присутствующих в обоих наборах.

Библиотека pandas при слиянии делает следующее:
1. В датафреймах она определяет столбцы с общими метками. Эти столбцы рассматриваются в качестве ключей для выполнения соединения.
2. Она создает новый объект DataFrame, столбцы которого — это метки на основе ключей, определенных на шаге 1, после них следуют все остальные метки обоих объектов, не являющиеся ключами.
3. Она сопоставляет значения в столбцах-ключах обоих объектов DataFrame.
4. В итоговом датафрейме она создает строку для каждого набора совпавших меток.
5. Затем она копирует данные из совпавших строк каждого объекта-источника в соответствующие строки и столбцы итогового датафрейма.
6. Итоговому датафрейму она присваивает новый тип индекса Int64Index.

Чтобы явно задать столбец, используемый для связывания объектов, можно воспользоваться параметром `on`. Если вы хотите выполнить слияние на основе столбцов с разными именами в каждом объекте, вы можете использовать параметры `left_on` и `right_on`. Чтобы выполнить слияние с помощью индексных меток строк обоих объектов DataFrame, можно воспользоваться параметрами `left_index=True` и `right_index=True`.

Поскольку в обоих объектах DataFrame присутствовал столбец с одинаковым именем, к именам столбцов в итоговом датафрейме добавляются суффиксы `_x` и `_y`, чтобы идентифицировать датафрейм источник, к которому относится данный столбец. Суффикс `_x` указывает, что столбец принадлежит датафрейму left, а суффикс `_y` указывает, что столбец принадлежит датафрейму right. Вы можете задать эти суффиксы с помощью параметра `suffixes` и передать последовательность из двух элементов.

In [7]:
# импортируем библиотеку datetime для работы с датами
from datetime import date
# это наши клиенты
customers = {'CustomerID': [10, 11],
             'Name': ['Mike', 'Marcia'],
             'Address': ['Address for Mike',
                         'Address for Marcia']}
customers = pd.DataFrame(customers)
# это наши заказы, сделанные клиентами,
# они связаны с клиентами с помощью столбца CustomerID
orders = {'CustomerID': [10, 11, 10],
          'OrderDate': [date(2014, 12, 1),
                        date(2014, 12, 1),
                        date(2014, 12, 1)]}
orders = pd.DataFrame(orders)
# выполняем слияние датафреймов customers и orders так, чтобы
# мы могли отправить товары
print(customers.merge(orders))

# создаем данные, которые будем использовать в качестве примеров
left_data = {'key1': ['a', 'b', 'c'],
             'key2': ['x', 'y', 'z'],
             'lval1': [0, 1, 2]}
right_data = {'key1': ['a', 'b', 'c'],
              'key2': ['x', 'a', 'z'],
              'rval1': [6, 7, 8]}
left = pd.DataFrame(left_data, index=[0, 1, 2])
right = pd.DataFrame(right_data, index=[1, 2, 3])
# демонстрируем слияние, не указывая столбцы, по которым
# нужно выполнить слияние
# этот программный код неявно выполняет слияние
# по всем общим столбцам
print(left.merge(right))
# демонстрируем слияние, явно задав столбец,
# по значениям которого нужно связать
# объекты DataFrame
print(left.merge(right, on='key1'))
# явно выполняем слияние с помощью двух столбцов
print(left.merge(right, on=['key1', 'key2']))

   CustomerID    Name             Address   OrderDate
0          10    Mike    Address for Mike  2014-12-01
1          10    Mike    Address for Mike  2014-12-01
2          11  Marcia  Address for Marcia  2014-12-01
  key1 key2  lval1  rval1
0    a    x      0      6
1    c    z      2      8
  key1 key2_x  lval1 key2_y  rval1
0    a      x      0      x      6
1    b      y      1      a      7
2    c      z      2      z      8
  key1 key2  lval1  rval1
0    a    x      0      6
1    c    z      2      8


3. Настройка семантики соединения при выполнении слияния. Привести **свои** примеры на **каждый тип** соединения..

**Ответ**: Тип соединения, выполняемый функцией `pd.merge()` по умолчанию — внутреннее соединение (inner join). Чтобы использовать другой тип соединения, укажите его с помощью параметра `how` функции `pd.merge()` (или метода `.merge()`). Допустимыми параметрами являются:
- `inner`: выполняет пересечение ключей из обоих объектов DataFrame
- `outer`: выполняет объединение ключей из обоих объектов DataFrame
- `left`: использует только ключи из левого объекта DataFrame
- `right`: использует только ключи из правого объекта DataFrame

Внутреннее соединение является значением по умолчанию и возвращает слияние данных из обоих объектов DataFrame только когда значения совпадают.

Внешнее соединение возвращает все строки, попавшие во внутреннее соединение, плюс все строки из датафрейма left, не попавшие во внутреннее соединение, плюс все строки из датафрейма right, не попавшие во внутреннее соединение. Отсутствующие значения заменяются значениями NaN.

Левое соединение возвращает все строки, попавшие во внутреннее соединение, плюс все строки из датафрейма left, не попавшие во внутреннее соединение (для которых не нашлось пары в датафрейме right).

Правое соединение возвращает все строки, попавшие во внутреннее соединение, плюс все строки из датафрейма right, не попавшие во внутреннее соединение (для которых не нашлось пары в датафрейме left).

Кроме того, библиотека pandas предлагает метод `.join()`, который можно использовать для выполнения соединения с помощью индексных меток двух объектов DataFrame (вместо значений столбцов). Если столбцы в обоих объектах DataFrame не имеют уникальных имен, вы должны указать суффиксы с помощью параметров `lsuffix` и `rsuffix`. Тип соединения, используемый по умолчанию — внешнее соединение. Это отличается от стандартного метода `.merge()`, в котором по умолчанию применяется внутреннее соединение.

In [8]:
left_data = {'key1': ['a', 'b', 'c'],
             'key2': ['x', 'y', 'z'],
             'lval1': [0, 1, 2]}
right_data = {'key1': ['a', 'b', 'c'],
              'key2': ['x', 'a', 'z'],
              'rval1': [6, 7, 8]}
left = pd.DataFrame(left_data, index=[0, 1, 2])
right = pd.DataFrame(right_data, index=[1, 2, 3])

# внутреннее соединение (по умолчанию)
print('inner join:')
print(left.merge(right, how='inner'))

# внешнее соединение
print('outer join:')
print(left.merge(right, how='outer'))

# левое соединение
print('left join:')
print(left.merge(right, how='left'))

# правое соединение
print('right join:')
print(left.merge(right, how='right'))

# метод .join() с суффиксами
print('join с суффиксами:')
print(left.join(right, lsuffix='_left', rsuffix='_right'))

inner join:
  key1 key2  lval1  rval1
0    a    x      0      6
1    c    z      2      8
outer join:
  key1 key2  lval1  rval1
0    a    x    0.0    6.0
1    b    a    NaN    7.0
2    b    y    1.0    NaN
3    c    z    2.0    8.0
left join:
  key1 key2  lval1  rval1
0    a    x      0    6.0
1    b    y      1    NaN
2    c    z      2    8.0
right join:
  key1 key2  lval1  rval1
0    a    x    0.0      6
1    b    a    NaN      7
2    c    z    2.0      8
join с суффиксами:
  key1_left key2_left  lval1 key1_right key2_right  rval1
0         a         x      0        NaN        NaN    NaN
1         b         y      1          a          x    6.0
2         c         z      2          b          a    7.0


4. Поворот данных для преобразования значений в индексы и наоборот. Привести **свои** примеры.

**Ответ**: Данные часто хранятся в состыкованном формате («в столбик»), который еще называют форматом записи. Он используется в базах данных, CSV-файлах и таблицах Excel. В состыкованном формате данные часто не нормированы и имеют повторяющиеся значения в нескольких столбцах или значения, которые логически должны относиться к другим таблицам (нарушая концепцию аккуратных данных).

Проблема такого представления данных заключается в том, что довольно трудно выделить показания, относящиеся к конкретной оси.

Проблема возникает, когда вам нужно узнать значения по всем осям в данный интервал времени, а не только по оси x. Чтобы получить значения по всем осям, можно выполнить отбор каждого значения оси, но это будет громоздкий код.

Более оптимальным станет такое представление данных, в котором столбцы будут представлять уникальные значения переменной axis. Чтобы преобразовать данные в этот формат, используйте метод `.pivot()` объекта DataFrame.

In [9]:
sensor_readings = pd.read_csv('accel.csv', sep=';')
print(sensor_readings)

# логический отбор значений по оси x
print(sensor_readings[sensor_readings['axis'] == 'x'])

# используем .pivot() для преобразования данных
pivoted = sensor_readings.pivot(index='interval',
                                columns='axis',
                                values='reading')
print(pivoted)

    interval axis  reading
0          0    x      0.0
1          0    y      0.5
2          0    z      1.0
3          1    x      0.1
4          1    y      0.4
5          1    z      0.1
6          2    x      0.4
7          2    y      0.3
8          2    z      0.8
9          3    x      0.3
10         3    y      0.2
11         3    z      0.7
   interval axis  reading
0         0    x      0.0
3         1    x      0.1
6         2    x      0.4
9         3    x      0.3
axis        x    y    z
interval               
0         0.0  0.5  1.0
1         0.1  0.4  0.1
2         0.4  0.3  0.8
3         0.3  0.2  0.7


5. Что такое состыковка и расстыковка данных?

**Ответ**: Методы `.stack()` и `.unstack()` схожи с методом `.pivot()`. Процесс состыковки поворачивает уровень меток столбцов, превращая его в индекс строк. Расстыковка выполняет обратное действие, то есть поворачивает уровень индекса строк, превращая его в индекс столбцов.

Одно из различий между состыковкой/расстыковкой и поворотом заключается в том, что в отличие от поворота операции состыковки и расстыковки могут повернуть конкретные уровни иерархического индекса. Кроме того, если поворот сохраняет одинаковое количество уровней индекса, состыковка и расстыковка всегда увеличивают количество уровней по индексу одной оси (количество столбцов для расстыковки и количество строк для состыковки) и уменьшают количество уровней по другой оси.

5.1 Состыковка с помощью неиерархических индексов. Привести **свои** примеры.

**Ответ**: Состыковка помещает уровень индекса столбцов в новый уровень индекса строк. Поскольку объект DataFrame имеет только один уровень, происходит сворачивание объекта DataFrame в объект Series с иерархическим индексом строк.

Чтобы посмотреть значения, нам нужно передать кортеж в индексатор объекта Series, который выполняет поиск с помощью индекса.

Если объект DataFrame содержит несколько столбцов, то все столбцы помещаются в один и тот же дополнительный уровень нового объекта.

Расстыковка выполняет противоположную операцию, помещая уровень индекса строк в уровень оси столбцов.

In [10]:
# создаем простой датафрейм с одним столбцом
df = pd.DataFrame({'a': [1, 2]}, index=['one', 'two'])
print(df)
# помещаем столбец в еще один уровень индекса строк
# результатом становится объект Series, в которой
# значения можно просмотреть с помощью мультииндекса
stacked1 = df.stack()
print(stacked1)
# ищем значение для 'one'/'a', передав кортеж в индексатор
print(stacked1[('one', 'a')])

# создаем датафрейм с двумя столбцами
df = pd.DataFrame({'a': [1, 2],
                   'b': [3, 4]},
                  index=['one', 'two'])
print(df)
stacked2 = df.stack()
print(stacked2)

     a
one  1
two  2
one  a    1
two  a    2
dtype: int64
1
     a  b
one  1  3
two  2  4
one  a    1
     b    3
two  a    2
     b    4
dtype: int64


5.2 Расстыковка с помощью иерархических индексов. Привести **свои** примеры.

**Ответ**: Расстыковка помещает самый внутренний уровень индекса строк в новый уровень индекса столбцов, в результате получаем столбцы с типом индекса MultiIndex.

Чтобы выполнить расстыковку другого уровня, используйте параметр `level`.

Для выполнения одновременной расстыковки нескольких уровней нужно передать в метод `.unstack()` список уровней. Кроме того, если уровни имеют имена, их можно перечислить с помощью имен, а не позиций.

Состыковка и расстыковка всегда помещают уровни в самые внутренние уровни другого индекса. При перемещении данных, осуществляемом в ходе состыковки и расстыковки (а также повороте), не происходит потери информации. При выполнении этих операций просто меняется способ организации и просмотра данных.

In [11]:
sensor_readings = pd.read_csv('accel.csv', sep=';')
# создаем две копии данных акселерометра,
# по одной для каждого пользователя
user1 = sensor_readings.copy()
user2 = sensor_readings.copy()
# добавляем столбец who в каждую копию
user1['who'] = 'Mike'
user2['who'] = 'Mikael'
# давайте отмасштабируем данные user2
user2['reading'] *= 100
# и организуем данные так, чтобы получить иерархический
# индекс строк
multi_user_sensor_data = pd.concat([user1, user2]) \
    .set_index(['who', 'interval', 'axis'])
print(multi_user_sensor_data)

# извлекаем показания, относящиеся к пользователю Mike,
# с помощью индекса
print(multi_user_sensor_data.loc['Mike'])

# извлекаем все показания по всем осям
# и по всем пользователям в интервале 1
print(multi_user_sensor_data.xs(1, level='interval'))

# выполняем расстыковку, в результате самый внутренний
# уровень индекса строк (уровень axis)
# стал уровнем индекса столбцов
print(multi_user_sensor_data.unstack())

# выполняем расстыковку по уровню 0
print(multi_user_sensor_data.unstack(level=0))

# выполняем расстыковку уровней who и axis
unstacked = multi_user_sensor_data.unstack(['who', 'axis'])
print(unstacked)

# и, конечно, мы можем выполнить состыковку уровней,
# которые расстыковали
# выполняем состыковку уровня who
print(unstacked.stack(level='who'))

                      reading
who    interval axis         
Mike   0        x         0.0
                y         0.5
                z         1.0
       1        x         0.1
                y         0.4
                z         0.1
       2        x         0.4
                y         0.3
                z         0.8
       3        x         0.3
                y         0.2
                z         0.7
Mikael 0        x         0.0
                y        50.0
                z       100.0
       1        x        10.0
                y        40.0
                z        10.0
       2        x        40.0
                y        30.0
                z        80.0
       3        x        30.0
                y        20.0
                z        70.0
               reading
interval axis         
0        x         0.0
         y         0.5
         z         1.0
1        x         0.1
         y         0.4
         z         0.1
2        x         0.4
         y   

6. Расплавление данных для преобразования «широкого» формата в «длинный» и наоборот. Привести **свои** примеры.

**Ответ**: Расплавление — это тип реорганизации данных, который часто называют преобразованием объекта DataFrame из «широкого» формата (wide format) в «длинный» формат (long format). Этот формат является общепринятым при проведении различных видов статистического анализа, и данные, которые вы считываете, могут быть представлены уже в расплавленном виде.

С технической точки зрения расплавление — это процесс изменения формы объекта DataFrame, в результате получаем формат, в котором два или более столбцов, названные `variable` и `value`, создаются путем сворачивания меток столбцов исходного датафрейма в столбец `variable`, а затем происходит перемещение значений из этих столбцов в соответствующую позицию столбца `value`. Все остальные столбцы превращаются в столбцы-идентификаторы, которые помогают описать данные.

Данные после расплавления реструктурированы, поэтому легко извлечь значение для любой комбинации переменных `variable` и `Name`. Кроме того, в таком формате представления проще добавить новую переменную и наблюдение, поскольку можно просто добавить данные в виде новой строки и для этого не требуется менять структуру датафрейма, добавляя новый столбец.

In [12]:
# создаем объект DataFrame с переменными Height и Weight
# и столбцом-идентификатором Name
data = pd.DataFrame({
    'Name': ['Mike', 'Mikael'],
    'Height': [180, 175],
    'Weight': [75, 70]
})
print(data)

# расплавляем датафрейм, используя столбец Name
# в качестве столбца-идентификатора,
# а столбцы Height и Weight в качестве переменных
melted = data.melt(id_vars=['Name'])
print(melted)

     Name  Height  Weight
0    Mike     180      75
1  Mikael     175      70
     Name variable  value
0    Mike   Height    180
1  Mikael   Height    175
2    Mike   Weight     75
3  Mikael   Weight     70


In [13]:
import subprocess
subprocess.run(["jupyter", "nbconvert", "--to", "html", __vsc_ipynb_file__])

[NbConvertApp] Converting notebook /Users/efim/dev/university-6sem-data-analysis/topic3/questions3/questions3.ipynb to html
[NbConvertApp] Writing 364196 bytes to /Users/efim/dev/university-6sem-data-analysis/topic3/questions3/questions3.html


CompletedProcess(args=['jupyter', 'nbconvert', '--to', 'html', '/Users/efim/dev/university-6sem-data-analysis/topic3/questions3/questions3.ipynb'], returncode=0)